# Connect VS Code to Colab GPU

This notebook sets up SSH access to your Colab runtime, allowing you to:
- Code in VS Code locally
- Execute on Colab's free GPU
- Use VS Code debugger with Colab resources

**Steps:**
1. Run cells in this notebook to setup SSH
2. Copy the connection URL
3. Connect from VS Code using Remote-SSH extension
4. Develop and train on Colab GPU!

## Step 1: Install Cloudflared (SSH Tunnel)

In [ ]:
%%bash
# Install cloudflared for tunneling
wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
sudo dpkg -i cloudflared-linux-amd64.deb
rm cloudflared-linux-amd64.deb
echo "✓ Cloudflared installed"

## Step 2: Setup SSH Server

In [ ]:
import subprocess
import getpass

# Set SSH password
print("Enter a password for SSH access:")
password = getpass.getpass()

# Set root password
subprocess.run(f"echo 'root:{password}' | sudo chpasswd", shell=True, check=True)

# Create SSH directory
!sudo mkdir -p /var/run/sshd

# Configure SSH
!sudo sed -i 's/#PermitRootLogin prohibit-password/PermitRootLogin yes/' /etc/ssh/sshd_config
!sudo sed -i 's/#PasswordAuthentication yes/PasswordAuthentication yes/' /etc/ssh/sshd_config

# Start SSH service
!sudo service ssh start

print("✓ SSH server configured and started")
print(f"Password set successfully (remember it!)")

## Step 3: Clone Repository

In [ ]:
# Clone your repository
!git clone https://github.com/AliABULIEL/AccelomtryFoundationModel.git
%cd AccelomtryFoundationModel

# Install dependencies
!pip install -q -r requirements.txt
!pip install -q -e .

print("✓ Repository cloned and dependencies installed")

## Step 4: Mount Google Drive (for checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/ttm_accelerometry'
os.makedirs(f"{DRIVE_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/data", exist_ok=True)

print(f"✓ Drive mounted at: {DRIVE_DIR}")

## Step 5: Start Cloudflared Tunnel

**IMPORTANT:** Keep this cell running! Copy the URL it provides.

In [ ]:
import subprocess
import threading
import time

print("="*70)
print("Starting Cloudflared tunnel...")
print("="*70)
print("\nWait for the URL to appear below...\n")

# Start cloudflared
proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'ssh://localhost:22'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True
)

# Print output in real-time
url_found = False
for line in iter(proc.stdout.readline, ''):
    print(line.rstrip())
    if 'trycloudflare.com' in line and not url_found:
        url = line.split('https://')[1].split()[0]
        print("\n" + "="*70)
        print("✓ TUNNEL READY!")
        print("="*70)
        print(f"\nConnection URL: https://{url}")
        print("\nTo connect from VS Code:")
        print("1. Install 'Remote - SSH' extension")
        print(f"2. Press F1 → 'Remote-SSH: Connect to Host'")
        print(f"3. Enter: root@{url}")
        print("4. Enter the password you set above")
        print("5. Open folder: /content/AccelomtryFoundationModel")
        print("="*70)
        url_found = True

## Alternative: Use ngrok (if cloudflared doesn't work)

In [ ]:
# Uncomment and run if cloudflared doesn't work

# # Install ngrok
# !pip install -q pyngrok

# from pyngrok import ngrok

# # Get ngrok token from https://dashboard.ngrok.com/get-started/your-authtoken
# ngrok_token = input("Enter your ngrok token: ")
# ngrok.set_auth_token(ngrok_token)

# # Start tunnel
# ssh_tunnel = ngrok.connect(22, "tcp")
# print(f"\nSSH Connection: {ssh_tunnel.public_url}")
# print("Use this in VS Code Remote-SSH")

## Test GPU Access

In [ ]:
import torch

print("GPU Status:")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    
    # Test allocation
    x = torch.randn(1000, 1000).cuda()
    print(f"  ✓ GPU test successful")
else:
    print("  ⚠ No GPU available. Change Runtime → Change runtime type → GPU")

## Keep-Alive (Prevent Colab disconnect)

Run this to keep the session alive while working in VS Code:

In [ ]:
import time
from IPython.display import clear_output

print("Keep-alive running... (Ctrl+C to stop)")
print("This prevents Colab from timing out while you work in VS Code")

try:
    while True:
        time.sleep(60)  # Ping every minute
        clear_output(wait=True)
        print(f"Keep-alive: {time.strftime('%Y-%m-%d %H:%M:%S')}")
        print("VS Code connection active...")
except KeyboardInterrupt:
    print("Keep-alive stopped")

## Usage from VS Code

Once connected, you can:

### Run Training
```bash
cd /content/AccelomtryFoundationModel
python scripts/train_vscode.py --data data/synthetic_data.h5 --quick
```

### Debug with VS Code Debugger
1. Press F5 in VS Code
2. Select "Train (Quick)" from launch configurations
3. Set breakpoints and debug!

### Run Tests
```bash
pytest tests/ -v
```

### Interactive Python
```python
import torch
print(torch.cuda.is_available())  # Should be True!
```